# 02 · Fortgeschritten: Dashboard, IDS und 3D

[![In Colab öffnen](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/LTplus-AG/bxmid03-hs26/blob/main/notebooks/2026-10-27_Automatisierung-I/02_Fortgeschritten_Dashboard_IDS_3D.ipynb)

**BXMid03 · HS26 · Automatisierung I**

Voraussetzung: Notebook 01 (Zellen ausführen, Modell wählen, Psets und Mengen).

In diesem Notebook:
1. das ganze Modell als **Tabelle** (eine Zeile pro Bauteil),
2. ein **Dashboard** mit Diagrammen,
3. ein **interaktiver Filter**,
4. eine **IDS-Prüfung**, die wir direkt in Python schreiben,
5. das Modell in **3D** – mit Klick auf ein Bauteil zurück in Python.

Für die 3D-Ansicht braucht der Browser **WebGPU** (Chrome oder Edge).

In [ ]:
#@title Installieren
!pip install -q ifcopenshell ifctester ifc-anywidget plotly
print("Fertig.")

## Schritt 1 · Modell wählen

Diese Zelle ist ein **Formular**: rechts erscheint eine Auswahlliste statt Code.
Wenn du eine andere Auswahl triffst, läuft die Zelle automatisch neu. Danach die folgenden Zellen erneut ausführen
(oder *Laufzeit → Alle ausführen*).

In [ ]:
#@title Modell wählen { run: "auto" }
#@markdown Wähle ein Beispielmodell aus der Liste. Die Zelle lädt es aus dem Kurs-Repository auf GitHub.
#@markdown „Eigene Datei hochladen“ öffnet einen Dateidialog – nur Modelle verwenden, die das Projekt verlassen dürfen.
modell = "Tragwerk · BIMcollab-Beispiel (IFC2X3)" #@param ["Tragwerk · BIMcollab-Beispiel (IFC2X3)", "Brücke · IfcBridge-Beispiel (IFC4X3)", "Hochhaus · Snowdon Towers Tragwerk (IFC2X3)", "Holzrahmenwand · fehlerhaft (IFC4)", "Holzrahmenwand · korrigiert (IFC4)", "Eigene Datei hochladen"]

import urllib.request

BASIS = "https://raw.githubusercontent.com/LTplus-AG/bxmid03-hs26/main/modelle/"
DATEIEN = {
    "Tragwerk · BIMcollab-Beispiel (IFC2X3)": "Team_5_BIMcollab_Example_STR.ifc",
    "Brücke · IfcBridge-Beispiel (IFC4X3)": "Team_1_Bruecke_Arbeitsmodell.ifc",
    "Hochhaus · Snowdon Towers Tragwerk (IFC2X3)": "Team_4_Snowdon_Structural_Baufortschritt.ifc",
    "Holzrahmenwand · fehlerhaft (IFC4)": "Demo_Holzrahmen_fehlerhaft.ifc",
    "Holzrahmenwand · korrigiert (IFC4)": "Demo_Holzrahmen_korrigiert.ifc",
}

if modell == "Eigene Datei hochladen":
    from google.colab import files
    pfad = list(files.upload())[0]          # Dateidialog; die Datei liegt danach auf dem Google-Server
else:
    pfad = DATEIEN[modell]
    urllib.request.urlretrieve(BASIS + pfad, pfad)   # Datei aus GitHub herunterladen

print("Modell bereit:", pfad)

## Schritt 2 · Das Modell als Tabelle

Wir sammeln für jedes Bauteil die wichtigsten Angaben in einer Zeile.
Mit so einer Tabelle lassen sich alle weiteren Fragen schnell beantworten: zählen, summieren, filtern, zeichnen.

In [ ]:
#@title Tabelle aller Bauteile
import ifcopenshell
import ifcopenshell.util.element as ue
import ifcopenshell.util.unit as uu
import pandas as pd

m = ifcopenshell.open(pfad)
laenge = uu.calculate_unit_scale(m)                 # Länge  -> m
volumen = uu.calculate_unit_scale(m, "VOLUMEUNIT")  # Volumen -> m³ (kann anders sein als Länge!)

def menge(element, name):
    """Sucht eine Menge (z. B. 'Width') in allen Mengen-Sets eines Bauteils.

    Mengen stehen je nach Software in 'Qto_WallBaseQuantities', 'BaseQuantities' o. ä.
    Wir suchen deshalb in allen Sets und nehmen den ersten Treffer.
    """
    for qset in ue.get_psets(element, qtos_only=True).values():
        if name in qset:
            return qset[name]
    return None

def material_name(b):
    """Gibt einen lesbaren Materialnamen zurück (Schichten werden zusammengefasst)."""
    mat = ue.get_material(b, should_skip_usage=True)
    if mat is None:
        return "ohne Material"
    if mat.is_a("IfcMaterial"):
        return mat.Name or "ohne Namen"
    namen = [x.Name for x in ue.get_materials(b) if x is not None and x.Name]
    return " / ".join(namen) if namen else mat.is_a()

def zahl(wert, faktor):
    return None if wert is None else round(wert * faktor, 4)

zeilen = []
for b in m.by_type("IfcElement"):
    if b.is_a("IfcOpeningElement"):
        continue
    geschoss = ue.get_container(b)
    typ = ue.get_type(b)
    zeilen.append({
        "GlobalId": b.GlobalId,
        "Klasse": b.is_a(),
        "Name": b.Name,
        "Typ": typ.Name if typ else None,
        "Geschoss": geschoss.Name if geschoss else "ohne Geschoss",
        "Material": material_name(b),
        "Länge [m]": zahl(menge(b, "Length"), laenge),
        "Breite [m]": zahl(menge(b, "Width"), laenge),
        "Höhe [m]": zahl(menge(b, "Height"), laenge),
        "Volumen [m³]": zahl(menge(b, "NetVolume") or menge(b, "GrossVolume"), volumen),
    })

df = pd.DataFrame(zeilen)
print(len(df), "Bauteile ·", df["Klasse"].nunique(), "Klassen ·", df["Geschoss"].nunique(), "Geschosse")
df.head(10)

## Schritt 3 · Dashboard

**Plotly** zeichnet interaktive Diagramme: mit der Maus über Balken fahren zeigt Werte,
Doppelklick in der Legende blendet Klassen ein/aus, die Kamera oben rechts speichert ein Bild.

In [ ]:
#@title Dashboard: Klassen, Geschosse, Material, Volumen
import plotly.express as px
from plotly.subplots import make_subplots

fig = make_subplots(rows=2, cols=2, subplot_titles=(
    "Bauteile je Klasse", "Bauteile je Geschoss", "Volumen je Material [m³]", "Volumen je Klasse [m³]"))

je_klasse = df["Klasse"].value_counts()
fig.add_bar(x=je_klasse.index, y=je_klasse.values, row=1, col=1, showlegend=False)

# Geschoss x Klasse als gestapelte Balken
je_geschoss = df.groupby(["Geschoss", "Klasse"]).size().unstack(fill_value=0)
for k in je_geschoss.columns:
    fig.add_bar(x=je_geschoss.index, y=je_geschoss[k], name=k, row=1, col=2)

vol_mat = df.groupby("Material")["Volumen [m³]"].sum().sort_values(ascending=False).head(10)
fig.add_bar(x=vol_mat.index, y=vol_mat.values, row=2, col=1, showlegend=False)

vol_kl = df.groupby("Klasse")["Volumen [m³]"].sum().sort_values(ascending=False)
fig.add_bar(x=vol_kl.index, y=vol_kl.values, row=2, col=2, showlegend=False)

fig.update_layout(barmode="stack", height=750, title=f"Modell: {pfad}")
fig.show()

In [ ]:
#@title Treemap: Klasse → Typ (Fläche = Anzahl)
baum = df.fillna({"Typ": "ohne Typ"}).groupby(["Klasse", "Typ"]).size().reset_index(name="Anzahl")
px.treemap(baum, path=["Klasse", "Typ"], values="Anzahl", height=500).show()

## Schritt 4 · Interaktiver Filter

`ipywidgets` macht aus einer Funktion eine kleine Oberfläche. Ändert sich die Auswahl,
läuft die Funktion neu: Tabelle und Histogramm passen sich an.

In [ ]:
#@title Filter nach Klasse und Geschoss
import ipywidgets as widgets
from IPython.display import display

def zeige(klasse, geschoss, groesse):
    teil = df[df["Klasse"] == klasse]
    if geschoss != "alle":
        teil = teil[teil["Geschoss"] == geschoss]
    print(f"{len(teil)} Bauteile")
    werte = teil[groesse].dropna()
    if len(werte):
        px.histogram(werte, nbins=20, title=f"{groesse} · {klasse}", height=300).show()
    else:
        print(f"Keine Werte für {groesse} – die Menge fehlt bei diesen Bauteilen.")
    display(teil.head(15))

widgets.interact(
    zeige,
    klasse=sorted(df["Klasse"].unique()),
    geschoss=["alle"] + sorted(df["Geschoss"].unique()),
    groesse=["Länge [m]", "Breite [m]", "Höhe [m]", "Volumen [m³]"],
);

## Schritt 5 · IDS in Python schreiben und prüfen

**IDS** (Information Delivery Specification) ist das buildingSMART-Format für Anforderungen.
Mit `ifctester` schreiben wir eine IDS direkt in Python, prüfen das Modell und speichern
die IDS als Datei – sie lässt sich danach z. B. in [ifclite.com](https://www.ifclite.com) wieder laden.

Unsere zwei Anforderungen an Wände:
1. `Pset_WallCommon.IsExternal` muss vorhanden sein (aussen/innen),
2. `Pset_WallCommon.FireRating` muss vorhanden sein (Feuerwiderstand).

In IFC2X3 heissen viele Wände `IfcWallStandardCase` – deshalb nennen wir beide Klassen.

In [ ]:
#@title IDS erstellen, prüfen, Bericht speichern
from ifctester import ids, reporter

waende = ids.Restriction(options={"enumeration": ["IFCWALL", "IFCWALLSTANDARDCASE"]})

anforderungen = ids.Ids(title="BXMid03 · Wände", author="bxmid03@example.org")
for eigenschaft, zweck in [("IsExternal", "Aussen- oder Innenwand"), ("FireRating", "Feuerwiderstand")]:
    spec = ids.Specification(name=f"Wände: {eigenschaft}", description=zweck,
                             ifcVersion=["IFC2X3", "IFC4", "IFC4X3_ADD2"])
    spec.applicability.append(ids.Entity(name=waende))                              # Auswahl
    spec.requirements.append(ids.Property(propertySet="Pset_WallCommon", baseName=eigenschaft))  # Anforderung
    anforderungen.specifications.append(spec)

anforderungen.validate(m)

for spec in anforderungen.specifications:
    print(f"{'✅' if spec.status else '❌'} {spec.name}: "
          f"{len(spec.applicable_entities)} geprüft, {len(spec.failed_entities)} durchgefallen")

anforderungen.to_xml("waende.ids")              # IDS-Datei
bericht = reporter.Html(anforderungen); bericht.report(); bericht.to_file("ids_bericht.html")
print("Gespeichert: waende.ids, ids_bericht.html (Ordner-Symbol links)")

In [ ]:
#@title Bericht hier anzeigen
from IPython.display import HTML
HTML(open("ids_bericht.html", encoding="utf-8").read())

## Schritt 6 · Das Modell in 3D

Das Community-Projekt [ifc-anywidget](https://github.com/sanzoghenzo/ifc-viewer-anywidget) (MIT-Lizenz) bringt den
**IFC-Lite-Viewer** ins Notebook.

- Maus links: drehen · rechts oder Umschalt: verschieben · Rad: zoomen
- **Klick auf ein Bauteil** wählt es aus. Die nächste Zelle zeigt seine Zeile aus unserer Tabelle.

`enable_custom_widget_manager()` erlaubt Colab, Widgets von Drittanbietern anzuzeigen.

In [ ]:
#@title 3D-Ansicht
from google.colab import output
output.enable_custom_widget_manager()

from ifc_anywidget import IfcViewer
viewer = IfcViewer(ifc_model=m)
viewer

In [ ]:
#@title Ausgewählte Bauteile anzeigen (nach dem Klick ausführen)
auswahl = viewer.selected_guids or []
if not auswahl:
    print("Noch nichts ausgewählt: im 3D-Fenster auf ein Bauteil klicken, dann diese Zelle erneut ausführen.")
df[df["GlobalId"].isin(auswahl)]

## Probiert selbst

1. Welches Geschoss hat das grösste Bauteilvolumen? (Tipp: `df.groupby("Geschoss")["Volumen [m³]"].sum()`)
2. Ergänze die IDS um eine dritte Anforderung, z. B. `LoadBearing`.
3. Lade `waende.ids` in [ifclite.com](https://www.ifclite.com) (*Analyze → Data validation*) und vergleiche.
4. Klicke im 3D-Fenster eine Stütze an. Welcher Typ ist es?

**Weiter:** Notebook 03 zeigt, was darüber hinaus möglich ist.